In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import random
import csv
import json
import time


import numpy as np
import config

import yaml

load_dotenv(find_dotenv())
engine = create_engine(f'postgresql://{config.db_username}:{config.db_password}@{config.db_host}:{config.db_port}/{config.db_name}')
connection = engine.connect()

## Define random options

In [2]:
def load_variables_sim(schema,year):
    with engine.connect() as connection:
        params = {}

        if year in [2003, 2013, 2023]:
            params['objid_list'] = [row[0] for row in connection.execute(text(f"SELECT DISTINCT objid FROM {schema}.photoobjall")).fetchall()]
            params['specobjid_list'] = [row[0] for row in connection.execute(text(f"SELECT DISTINCT specobjid FROM {schema}.specobjall")).fetchall()]

        if year in [2003, 2023]:
            z_mean, z_sd = connection.execute(text(f"SELECT AVG(z), STDDEV(z) FROM {schema}.photoz")).fetchone()
            params['z_mean'] = z_mean
            params['z_sd'] = z_sd

        if year in [2013, 2023]:
            params['fieldid_list'] = [row[0] for row in connection.execute(text(f"SELECT DISTINCT fieldid FROM {schema}.field")).fetchall()]

        if year == 2013:
            ra_st_mean, ra_st_sd, dec_st_mean, dec_st_sd = connection.execute(text(f"SELECT AVG(ra), STDDEV(ra), AVG(dec), STDDEV(dec) FROM {schema}.photoobjall")).fetchone() # FIXME previously was on stars
            params['ra_st_mean'] = ra_st_mean
            params['ra_st_sd'] = ra_st_sd
            params['dec_st_mean'] = dec_st_mean
            params['dec_st_sd'] = dec_st_sd
            # params['name_list'] = [row[0] for row in connection.execute(text(f"SELECT DISTINCT name FROM {schema}.DBObjects")).fetchall()]
            # params['type_list'] = [row[0] for row in connection.execute(text(f"SELECT DISTINCT type FROM {schema}.DBObjects")).fetchall()]
            # params['ta_list'] = [(row[0], row[1]) for row in connection.execute(text(f"SELECT DISTINCT type, access FROM {schema}.DBObjects")).fetchall()]

        if year == 2023:
            # params['mangaid_daptype_list'] = [(row[0], row[1]) for row in connection.execute(text(f"SELECT DISTINCT drp.mangaid, dap.daptype FROM {schema}.mangadrpall AS drp JOIN {schema}.mangadapall AS dap ON dap.mangaid = drp.mangaid")).fetchall()]
            params['pmf_list'] = [(row[0], row[1], row[2]) for row in connection.execute(text(f"SELECT DISTINCT s.plate, s.mjd, s.fiberid FROM {schema}.photoobjall AS p JOIN {schema}.specobjall s ON p.objid = s.bestobjid")).fetchall()]
            ra_p_mean, ra_p_sd, dec_p_mean, dec_p_sd = connection.execute(text(f"SELECT AVG(ra), STDDEV(ra), AVG(dec), STDDEV(dec) FROM {schema}.photoobjall")).fetchone()
            ra_s_mean, ra_s_sd, dec_s_mean, dec_s_sd = connection.execute(text(f"SELECT AVG(ra), STDDEV(ra), AVG(dec), STDDEV(dec) FROM {schema}.specobjall")).fetchone()
            dered_r_mean, dered_r_sd = connection.execute(text(f"SELECT AVG(dered_r), STDDEV(dered_r) FROM {schema}.photoobjall")).fetchone()
            params['ra_p_mean'] = ra_p_mean
            params['ra_p_sd'] = ra_p_sd
            params['dec_p_mean'] = dec_p_mean
            params['dec_p_sd'] = dec_p_sd
            params['ra_s_mean'] = ra_s_mean
            params['ra_s_sd'] = ra_s_sd
            params['dec_s_mean'] = dec_s_mean
            params['dec_s_sd'] = dec_s_sd
            params['dered_r_mean'] = dered_r_mean
            params['dered_r_sd'] = dered_r_sd

        return params
    
# YML version
def load_yml(file_path):
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data

def read_query_templates(query_file_name, type_exp):
    queries = []
    weights = []
    params_query_temp = []

    data = load_yml(f'../queries/{query_file_name}.yml')
    for row in data:
        weights.append(float(row['weight']))
        queries.append(row['query'])
        params_query_temp.append(row['params'].split(","))

    return queries, weights, params_query_temp

# CSV version
# def read_query_templates(query_file_name, type_exp):
#     queries = []
#     weights = []
#     params_query_temp = []
#     with open(f'../queries/{query_file_name}.csv', mode='r', encoding='utf-8') as csvfile:
#         csvreader = csv.reader(csvfile)
#         for row in csvreader:
#             weights.append(float(row[0]))
#             queries.append(row[1])
#             params_query_temp.append(json.loads(row[2]))
#     return queries, weights, params_query_temp
 
def get_sample_as_string(items, n):
    sample_size = min(n, len(items))
    sampled_items = random.sample(items, sample_size)
    result = ', '.join(f"{str(item)}" for item in sampled_items)
    return result

def get_random_interval(mean, std_dev):
    random_value1 = np.random.normal(mean, std_dev)
    random_value2 = np.random.normal(mean, std_dev)

    return sorted([random_value1, random_value2])

def simulate(num_sims, params, queries, weights, params_query_temp, db_name_for_queries):
    log_entries = []
    n_queries = len(queries)
    for num_exp in range(num_sims):
        chosen_index = random.choices(range(n_queries), weights=weights, k=1)[0]
    # for chosen_index in range(n_queries):
    #     print(chosen_index+1)
        query_params = {}
        for param in params_query_temp[chosen_index]:
            # Photoobjall
            if param == 'objid':
                query_params['objid'] = random.choice(params['objid_list'])
            elif param == 'ra_p':
                ra1, ra2 = get_random_interval(params['ra_p_mean'], params['ra_p_sd'])
                query_params['ra1'] = ra1
                query_params['ra2'] = ra2
            elif param == 'dec_p':
                dec1, dec2 = get_random_interval(params['dec_p_mean'], params['dec_p_sd'])
                query_params['dec1'] = dec1
                query_params['dec2'] = dec2
            elif param == 'objidlist':
                n = random.randint(1, 5000)
                objid_param = get_sample_as_string(params['objid_list'], n)
                query_params['objidlist'] = objid_param
            # Specobjall
            elif param == 'specobjid':
                query_params['specobjid'] = random.choice(params['specobjid_list'])
            elif param == 'ra_s':
                ra1, ra2 = get_random_interval(params['ra_s_mean'], params['ra_s_sd'])
                query_params['ra1'] = ra1
                query_params['ra2'] = ra2
            elif param == 'dec_s':
                dec1, dec2 = get_random_interval(params['dec_s_mean'], params['dec_s_sd'])
                query_params['dec1'] = dec1
                query_params['dec2'] = dec2
            elif param == 'fieldid':
                query_params['fieldid'] = random.choice(params['fieldid_list'])
            elif param == 'fieldidlist':
                n = random.randint(1, 100)
                fieldid_param = get_sample_as_string(params['fieldid_list'], n)
                query_params['fieldidlist'] = fieldid_param
            # Galaxy
            elif param == 'dered_r':
                dered_r1, dered_r2 = get_random_interval(params['dered_r_mean'], params['dered_r_sd'])
                query_params['dered_r1'] = dered_r1
                query_params['dered_r2'] = dered_r2
            # Photoz
            elif param == 'z':
                z1, z2 = get_random_interval(params['z_mean'], params['z_sd'])
                query_params['z1'] = z1
                query_params['z2'] = z2
            elif param == 'z1':
                query_params['z1'] = np.random.normal(params['z_mean'], params['z_sd'])
            # Mangadrpall, Mangadapall
            elif param == 'mangaid-daptype':
                random_item = random.choice(params['mangaid_daptype_list'])
                query_params['mangaid'] = random_item[0]
                query_params['daptype'] = random_item[1]
            # ["plate-mjd-fiberid"]
            elif param == 'plate-mjd-fiberid':
                random_item = random.choice(params['pmf_list'])
                query_params['plate'] = random_item[0]
                query_params['mjd'] = random_item[1]
                query_params['fiberid'] = random_item[2]
            # DBObjects
            elif param == 'name':
                random_item = random.choice(params['name_list'])
                query_params['name'] = random_item
            elif param == 'type':
                random_item = random.choice(params['type_list'])
                query_params['type'] = random_item
            elif param == 'type-access':
                random_item = random.choice(params['ta_list'])
                query_params['type'] = random_item[0]
                query_params['access'] = random_item[1]
            elif param == 'ra_st':
                ra_st1, ra_st2 = get_random_interval(params['ra_st_mean'], params['ra_st_sd'])
                query_params['ra_st1'] = ra_st1
                query_params['ra_st2'] = ra_st2
            elif param == 'dec_st':
                dec_st1, dec_st2 = get_random_interval(params['dec_st_mean'], params['dec_st_sd'])
                query_params['dec_st1'] = dec_st1
                query_params['dec_st2'] = dec_st2

        pre_query = queries[chosen_index].format(**query_params)
        final_query = "EXPLAIN (ANALYZE, FORMAT JSON) " + pre_query

        # if "instrument" in pre_query:
        #     print(pre_query)

        with engine.connect() as connection:
            connection.execute(text('SET search_path TO ' + db_name_for_queries))
            result = connection.execute(text(final_query))
            rows = result.fetchone()

        log_entry = rows[0][0]
        log_entry['QueryType'] = (chosen_index + 1)
        log_entry['NumRows'] = rows[0][0]["Plan"]["Actual Rows"]
        log_entry['FullQuery'] = pre_query
        log_entry['QueryParams'] = query_params

        log_entries.append(log_entry)

        # Print a message every 100 experiments
        if (num_exp + 1) % 100 == 0:
            print(f".", end="")
        if (num_exp + 1) % 1000 == 0:
            print(f" Completed {num_exp + 1} queries at {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())}")

    log_df = pd.DataFrame(log_entries)
    return log_df

def simulate_workload(db_name_for_variables, db_name_for_queries, year, num_sims, type_exp, query_file_name):
    # print(f"Simulation of {db_name}")
    params = load_variables_sim(db_name_for_variables, year)
    queries, weights, params_query = read_query_templates(query_file_name, type_exp)
    log_df = simulate(num_sims, params, queries, weights, params_query, db_name_for_queries)
    return log_df

## Execute simulations

In [5]:
db_name_for_variables = 'sdss_relational_2x' #sdss_relational2, sdss_relational_2x
scale = 2

scale_name = ''
if scale>1:
    scale_name = f'_{scale}x'


db_conf = []
db_conf.append({ 'year': 2003, 'query_file_name': '2003', 'db_name_for_queries': f'aa_sdss2003{scale_name}'})

db_conf.append({ 'year': 2003, 'query_file_name': '2003-optimized', 'db_name_for_queries': f'aa_sdss2003{scale_name}_optimized'})
db_conf.append({ 'year': 2013, 'query_file_name': '2013-over2003optimized', 'db_name_for_queries': f'aa_sdss2003{scale_name}_optimized'})
db_conf.append({ 'year': 2023, 'query_file_name': '2023-over2003optimized', 'db_name_for_queries': f'aa_sdss2003{scale_name}_optimized'})

db_conf.append({ 'year': 2013, 'query_file_name': '2013-optimized', 'db_name_for_queries': f'aa_sdss2013{scale_name}_optimized'})
db_conf.append({ 'year': 2023, 'query_file_name': '2023-over2013optimized', 'db_name_for_queries': f'aa_sdss2013{scale_name}_optimized'})

db_conf.append({ 'year': 2023, 'query_file_name': '2023-optimized', 'db_name_for_queries': f'aa_sdss2023{scale_name}_optimized'})

num_sims = 2000

type_exp = 'not-used'

outdir = (f"../results")
if not os.path.exists(outdir):
    os.mkdir(outdir)

for i in range(len(db_conf)):

    start_time = time.time()
    print(f'** RUNNING {db_conf[i]['query_file_name']}{scale_name} ({num_sims} queries) **')
    print('Started at',time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
    log_df = simulate_workload(db_name_for_variables, db_conf[i]['db_name_for_queries'], db_conf[i]['year'], num_sims, type_exp, db_conf[i]['query_file_name'])
    log_df.to_csv(f"{outdir}/results{scale_name}_{db_conf[i]['query_file_name']}.csv",index=False)
    print('Finished at',time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))


** RUNNING 2003_2x (2000 queries) **
Started at 2024-12-02 12:04:10


ProgrammingError: (psycopg2.errors.UndefinedTable) relation "photoobjall" does not exist
LINE 2: FROM PhotoObjAll p 
             ^

[SQL: EXPLAIN (ANALYZE, FORMAT JSON) SELECT p.value->>'run', p.value->>'type', p.value->>'ra', p.value->>'dec', p.value->>'g', p.value->>'r', p.value->>'err_g', p.value->>'err_r' 
FROM PhotoObjAll p 
WHERE (p.value->>'mode')::int8=1 AND p.key IN (1237648722299256975, 25842, 1237646647827300684, 1237662335721668986, 1237662535434306152, 1237680296739734125, 18286, 1237662535434174621, 1237651714801992328, 1237662535433453654, 1237662535434633900, 64552, 1237651800161255641, 35733, 1237662534898942263, 1237663531863638207, 1237645878494167404, 1237660979578667167, 1237648675070083222, 1237662534898352315, 24359, 13023, 1237663531339415632, 7568, 61675, 1237646645693711064, 33172, 1237657117866722706, 17678, 1237656567578035948, 33718, 8195, 1237648720143450228, 1237663531862655843, 1237648721765597248, 64023, 59687, 1237663531340464418, 86957, 1237674648855